In [ ]:
'''
 Models to take 
 ProtT5-XL-U50
'''

In [1]:
import transformers
print(transformers.__version__)

import torch
import re
from transformers import T5Tokenizer, T5EncoderModel

# ---- Load model and tokenizer ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "Rostlab/ProstT5" #"Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)
model = model.to(device)
model = model.eval()



4.57.3


2025-12-08 14:12:47.070885: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

In [2]:

# ---- Input sequences ----
sequence_examples = ["PRTEINO", "SEQWENCE"]

# Replace rare amino acids and add whitespace between letters
sequence_examples = [
    " ".join(list(re.sub(r"[UZOB]", "X", seq)))
    for seq in sequence_examples
]

# ---- Tokenize ----
ids = tokenizer.batch_encode_plus(
    sequence_examples,
    add_special_tokens=True,
    padding="longest"
)

input_ids = torch.tensor(ids["input_ids"]).to(device)
attention_mask = torch.tensor(ids["attention_mask"]).to(device) #batch_size, seq_len | 1:real , 0:padding

# ---- Get embeddings ----
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    emb = outputs.last_hidden_state   # shape: (batch_size, seq_len, 1024)

# ---- Mean pooling per protein ----
# Mask out padding tokens | attention_mask.unsqueeze(-1) ->[batch_size, seq_len, 1] (add Dim at the end)
#masked_emb : Real tokens (1) → emb * 1 | Padding tokens (0) → emb * 0 (Embedding becomes 0 for padded tokens)
#this is known as : masking out padding tokens
masked_emb = emb * attention_mask.unsqueeze(-1) #elementwise multiplication: broadcasting | [B, Seq_Len, Dim]

# Compute mean along sequence dimension |
# attention_mask.sum(dim=1, keepdim=True) give number of real tokens(non padding tokens) 
#attention_mask: [B, Seq_len]  ->[B, 1] , Keep dim=1, retains size 1
emb_per_protein = masked_emb.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)

print("Per-protein embeddings shape:", emb_per_protein.shape)
print("Embedding for first sequence:", emb_per_protein[0].shape)



Per-protein embeddings shape: torch.Size([2, 1024])
Embedding for first sequence: torch.Size([1024])


In [3]:
 emb_per_protein[0]

tensor([-0.0455, -0.0092, -0.0040,  ...,  0.1527, -0.0195, -0.0136],
       device='cuda:0')